# 02.02 语音交互基础

## 本节概述

<table style="text-align: left; margin-left: 0;">
<tr><td align="left"><b>前置要求</b></td><td align="left">已完成 02.01，准备好华为云凭证</td></tr>
<tr><td align="left"><b>本节目标</b></td><td align="left">理解音频数字表示，掌握 wav 播放与麦克风录音</td></tr>
<tr><td align="left"><b>本节内容</b></td><td align="left">音频数字化 → .env 凭证配置 → 音频播放 → 麦克风录音</td></tr>
</table>

## 第一部分：音频基础与播放


In [ ]:
# ===== 安装基础依赖 =====
# 注意：PyPI 包名是 SpeechRecognition（驼峰），不是 speech_recognition
# 拆开安装，避免单个失败拖累全部
import subprocess

# 基础通信库
!pip install setuptools requests websocket-client -q

# 语音识别 + 环境变量加载（★包名是驼峰 SpeechRecognition）
!pip install SpeechRecognition python-dotenv -q

# 华为云 OCR SDK + OpenAI 客户端（调 DeepSeek 用）
!pip install huaweicloudsdkocr openai -q
# 💡 华为云 OCR SDK 不兼容 Python 3.12+；当前 Python 3.11.4 (CANN) 内核满足要求

# opencv-python 与 numpy：02.05 生成 OCR 测试图片时需要（云环境也要）
!pip install opencv-python numpy -q

# ===== 检测云环境：决定是否装 playsound =====
import sys
is_cloud = False
try:
    from IPython import get_ipython
    is_cloud = get_ipython() is not None
except ImportError:
    pass

if not is_cloud:
    # 本地脚本环境才装 playsound（云环境装不上且用不到）
    !pip install playsound -q
    print("✅ 本地环境：已安装 playsound")
else:
    print("✅ 云环境：跳过 playsound（用 IPython Audio 播放，见方法 2）")

# ===== 验证关键依赖已装好 =====
import importlib
critical_pkgs = {
    'dotenv': 'python-dotenv',
    'speech_recognition': 'SpeechRecognition',
    'requests': 'requests',
}
missing = []
for mod, pip_name in critical_pkgs.items():
    try:
        importlib.import_module(mod)
    except ImportError:
        missing.append(pip_name)

if missing:
    print(f"⚠️ 以下包未装上，请手动执行: pip install {' '.join(missing)}")
else:
    print("✅ 关键依赖验证通过")
print("✅ 基础依赖安装完成")

## 1. 加载华为云凭证（安全实践）

华为云服务的调用需要 **AK（Access Key）** 和 **SK（Secret Key）** 凭证。**绝对不要把凭证硬编码在代码里**（尤其提交到 git 时会泄露）。

正确做法：把凭证写在 `.env` 文件中（已加入 `.gitignore` 不提交），用 `python-dotenv` 加载。

### 🔑 凭证获取指南

你需要 4 个凭证，按下表去对应平台申请：

<table style="text-align: left; margin-left: 0;">
<tr style="background-color:#f0f0f0">
  <th align="left">凭证</th><th align="left">用途</th><th align="left">去哪申请</th></tr>
<tr>
<td align="left"><b>华为云 AK</b><br><code>HUAWEI_SIS_AK</code></td>
<td align="left">华为云服务访问密钥 ID</td>
<td align="left">登录 <a href="https://console.huaweicloud.com">华为云控制台</a> → 右上角头像 → <b>「我的凭证」</b> → 「访问密钥」tab → 「新增访问密钥」<br>（需短信验证，下载的 <code>credentials.csv</code> 里有 AK）</td>
</tr>
<tr>
<td align="left"><b>华为云 SK</b><br><code>HUAWEI_SIS_SK</code></td>
<td align="left">华为云服务访问密钥（私钥）</td>
<td align="left">同上，<code>credentials.csv</code> 里的 Secret Access Key（<b>只在创建时显示一次，务必保存</b>）</td>
</tr>
<tr>
<td align="left"><b>Project ID</b><br><code>HUAWEI_SIS_PROJECT_ID</code></td>
<td align="left">项目唯一标识（每个区域一个）</td>
<td align="left">「我的凭证」页面 → <b>「项目列表」</b> tab → 找 <code>cn-east-3</code>（华东-上海三）对应的 <b>项目ID</b>（32位十六进制）</td>
</tr>
<tr>
<td align="left"><b>DeepSeek API Key</b><br><code>DEEPSEEK_API_KEY</code></td>
<td align="left">大语言模型调用（02.05 用）</td>
<td align="left">访问 <a href="https://platform.deepseek.com">DeepSeek 开放平台</a> → 注册登录 → <b>「API Keys」</b> → 「创建 API Key」<br>（新用户有免费额度；也可换成智谱、通义千问等兼容 OpenAI 格式的 API）</td>
</tr>
</table>

### ⚠️ 前置条件

申请凭证前，请先确认已开通这两个华为云服务（**新用户有免费额度**）：
- **语音交互服务 SIS**：https://console.huaweicloud.com/sis （02.03 ASR / 02.04 TTS 用）
- **文字识别服务 OCR**：https://console.huaweicloud.com/ocr （02.05 用）

> 💡 **费用提示**：华为云 SIS/OCR 和 DeepSeek 都有免费额度，本课程实验消耗的调用量很小（几十次），基本不会产生费用。

### 📝 填写规范

填到 `.env` 时，**等号右边直接写值，不加引号、不加空格**：
```ini
HUAWEI_SIS_AK=ABCD1234EFGH5678
HUAWEI_SIS_SK=abc123def456ghi789jkl
HUAWEI_SIS_REGION=cn-east-3
HUAWEI_SIS_PROJECT_ID=0a1b2c3d4e5f6789abcdef0123456789
DEEPSEEK_API_KEY=sk-abc123def456789ghi012jkl345mno678pqr
```

In [ ]:
# 创建 .env 文件模板（请替换成你自己的 AK/SK）
import os

env_content = """# 华为云凭证（请替换为你的真实值，切勿提交此文件）
HUAWEI_SIS_AK=你的AccessKey
HUAWEI_SIS_SK=你的SecretKey
HUAWEI_SIS_REGION=cn-east-3
HUAWEI_SIS_PROJECT_ID=你的项目ID（从我的凭证页面获取）
# DeepSeek API Key（02.05 LLM 集成用）
DEEPSEEK_API_KEY=你的DeepSeek_API_Key
"""

# 如果 .env 已存在，不覆盖（保留学习者已填的真实凭证）
if os.path.exists(".env"):
    print("ℹ️ .env 已存在，保留你已填写的凭证，跳过创建")
    print("   如需重新创建，请先手动删除 .env")
else:
    with open(".env", "w") as f:
        f.write(env_content)
    print("✅ 已创建 .env 模板")
    print()
    print("📋 接下来请编辑 .env，填入真实凭证：")
    print("   1. 在左侧文件树找到 .env（在 02_speech_ocr 目录下）")
    print("   2. 双击打开，把等号右边的占位符替换成真实值")
    print("   3. 保存（Ctrl+S）")
    print()
    print("🔑 凭证获取：")
    print("   - 华为云 AK/SK：控制台 → 我的凭证 → 访问密钥")
    print("   - Project ID：我的凭证 → 项目列表 → cn-east-3 的 Project ID")
    print("   - DeepSeek API Key：https://platform.deepseek.com → API Keys")
    print()
    print("⚠️ .env 含敏感信息，已被 .gitignore 排除，不会被提交到 git")

In [ ]:
# 加载环境变量并验证
import os
from dotenv import load_dotenv
load_dotenv()

ak = os.getenv('HUAWEI_SIS_AK', '')
sk = os.getenv('HUAWEI_SIS_SK', '')
region = os.getenv('HUAWEI_SIS_REGION', 'cn-east-3')
project_id = os.getenv('HUAWEI_SIS_PROJECT_ID', '')

# 安全检查：只显示前几位，不泄露完整凭证
print(f"AK: {ak[:6]}..." if len(ak) > 6 else "AK: 未配置")
print(f"SK: {'已配置' if sk else '未配置'}")
print(f"Region: {region}")
print(f"Project ID: {project_id[:8]}..." if len(project_id) > 8 else "Project ID: 未配置（SIS必需）")


## 2. 播放 wav 音频（方法 1：playsound，仅本地）

`playsound` 是本地最简单的播放方式，一行代码即可。**云环境（CANNLab / Jupyter）用不了**（无系统音频设备 + Python 3.11 装不上），云环境请直接用方法 2（IPython Audio）。

> 💡 下方代码会**自动检测环境**：云环境直接跳过并提示，本地环境才真正调用 playsound。

In [ ]:
# ===== 确保 16k16bit.wav 存在（不存在则用代码生成）=====
import os, numpy as np, wave, struct
wav_path = "./images/16k16bit.wav"
if not os.path.exists(wav_path):
    os.makedirs("./images", exist_ok=True)
    print("生成测试音频 16k16bit.wav（一段简单的提示音）...")
    sample_rate = 16000
    duration = 2  # 2秒
    t = np.linspace(0, duration, int(sample_rate * duration), endpoint=False)
    # 生成一段简单的提示音（440Hz + 880Hz）
    audio = (0.3 * np.sin(2 * np.pi * 440 * t) + 0.1 * np.sin(2 * np.pi * 880 * t))
    audio = (audio * 32767).astype(np.int16)
    with wave.open(wav_path, 'w') as wf:
        wf.setnchannels(1)
        wf.setsampwidth(2)
        wf.setframerate(sample_rate)
        wf.writeframes(audio.tobytes())
    print("✅ 已生成 16k16bit.wav")
else:
    print("✅ 本地已有 16k16bit.wav")

# 方法 1: playsound（仅本地 Windows/Linux/Mac 脚本环境）
# 云环境（CANNLab / Jupyter）会自动跳过，请用下方方法 2 的 IPython Audio
import sys

# 检测是否在云/Jupyter 环境
is_cloud = False
try:
    from IPython import get_ipython
    is_cloud = get_ipython() is not None
except ImportError:
    pass

audio_path = "./images/16k16bit.wav"

if is_cloud:
    print("☁️ 当前是云环境（CANNLab / Jupyter），跳过 playsound 播放")
    print("💡 云环境请用下方方法 2: IPython Audio（会显示内嵌播放器）")
else:
    try:
        from playsound import playsound
        print(f"正在播放: {audio_path}")
        playsound(audio_path)
        print("✅ 播放完成")
    except ImportError:
        print("⚠️ 未安装 playsound，本地运行可执行: pip install playsound")
    except Exception as e:
        print(f"播放失败: {e}")

## 3. 播放 wav 音频（方法 2：IPython Audio）

在 Jupyter Notebook 中，`IPython.display.Audio` 是更可靠的方式——它会在 cell 输出区域生成一个**内嵌播放器**，无需系统音频设备也能看到播放控件（云环境友好）：

In [ ]:
# 方法 2: IPython Audio（Mac / Jupyter 推荐）
from IPython.display import Audio, display

print("下面会出现一个音频播放器：")
display(Audio("./images/16k16bit.wav", autoplay=False))
print("💡 点击播放按钮即可试听（云环境也能显示播放器）")


### 两种播放方式对比

<table style="text-align: left; margin-left: 0;">
<tr style="background-color:#f0f0f0">
  <th align="left">方式</th><th align="left">优点</th><th align="left">缺点</th><th align="left">适用场景</th></tr>
<tr><td align="left"><code>playsound</code></td><td align="left">一行代码，简单</td><td align="left">依赖系统音频，云环境无效</td><td align="left">本地脚本</td></tr>
<tr><td align="left"><code>IPython Audio</code></td><td align="left">Jupyter 原生，云环境可见播放器</td><td align="left">只在 Notebook 中有效</td><td align="left">Jupyter/云环境</td></tr>
</table>

> 💡 本课程后续在云环境统一推荐 `IPython Audio`；本地脚本可用 `playsound`。

---

## 本节练习

**练习 1（选择）**：华为云 SIS 要求的音频格式是？
- A. 44.1kHz 采样率、16bit、立体声
- B. 16kHz 采样率、16bit、单声道
- C. 8kHz 采样率、8bit、单声道
- D. 48kHz 采样率、24bit、立体声

**练习 2（填空）**：AK/SK 凭证应该存放在 ______ 文件中，用 ______ 库加载，**不要** ______。

**练习 3（简答）**：为什么CANNLab 云环境用 `playsound` 播放会失败？应该用什么替代？

> 💡 参考答案见下方 code cell。

In [ ]:
# 查看本节练习答案
!cat ./answer/02.02_speech_basics/answers_playback.txt



---

## 第二部分：麦克风录音（需本地环境）

> ⚠️ 麦克风录音需要本地硬件，云环境无法运行。


In [ ]:
# ⚠️ 本代码需要本地麦克风，云环境会报错
# 本地运行时取消下方注释

# import speech_recognition as sr
#
# r = sr.Recognizer()      # 识别器实例
# mic = sr.Microphone()    # 自动检测默认麦克风
#
# # 列出所有可用麦克风（调试用）
# print("可用的麦克风设备:")
# for i, name in enumerate(sr.Microphone.list_microphone_names()):
#     print(f"  [{i}] {name}")

print("⚠️ 本节需要本地麦克风。云环境请看说明，本地运行时取消上方注释。")
print("本地运行后，会列出系统所有音频输入设备，[0] 通常是默认麦克风。")


## 1. 录音并保存

录音的核心流程：

```text
打开麦克风 → 校准环境噪声（1秒） → 录音（用户说话） → 转换为 16k16bit → 保存 wav
```

**关键参数**：

- `convert_rate=16000`：转换采样率为 16kHz（华为云 SIS 要求）；
- `convert_width=2`：转换位深为 16bit（每个采样点 2 字节）；
- `adjust_for_ambient_noise()`：适应环境噪声，**只在初始化时调用一次**。

In [ ]:
# ⚠️ 本代码需要本地麦克风
# 本地运行时取消注释，对着麦克风说一句话

# import speech_recognition as sr
# import wave
#
# r = sr.Recognizer()
# mic = sr.Microphone()
#
# with mic as source:
#     print("正在校准环境噪声...（请保持安静 1 秒）")
#     r.adjust_for_ambient_noise(source, duration=1)
#     print("校准完成！现在请对着麦克风说话...")
#
#     # 录音（phrase_time_limit 限制单次最长 5 秒）
#     audio = r.listen(source, phrase_time_limit=5)
#     print("录音完成！")
#
# # 转换为华为云要求的 16k16bit 格式
# wav_data = audio.get_wav_data(convert_rate=16000, convert_width=2)
#
# # 保存为文件
# with open("temp_command.wav", "wb") as f:
#     f.write(wav_data)
# print("✅ 已保存为 temp_command.wav (16kHz, 16bit)")

# ===== 云环境替代方案：用预置音频模拟录音产物 =====
# 云环境无麦克风，直接用预置的 16k16bit wav 代替录音
# 这样后续 ASR 识别流程完全可走通
from shutil import copyfile
copyfile('./images/clone_example_16k.wav', './temp_command.wav')
print('✅ 已用预置音频模拟录音产物: ./temp_command.wav')
print('💡 本地环境下，取消上方注释用真实麦克风录音；云环境用此预置音频即可')

print("⚠️ 本代码需本地麦克风，云环境无法运行。")
print("本地运行后会生成 temp_command.wav，可后续用于 ASR 识别。")


### 噪声校准为什么只调用一次

`adjust_for_ambient_noise()` 会采样当前环境噪声作为基线，用于后续录音时判断"什么时候开始说话"和"什么时候说完"。如果每次录音都调用，会**抹掉用户说话的开头部分**。正确做法是：程序启动时校准一次，之后循环录音复用同一基线。

## 2. 云环境替代方案

在云环境中，虽然没有麦克风，但你可以直接用本课程提供的 `16k16bit.wav`（一段预录音频）来体验后续的 ASR 识别流程——把"录音"这一步替换为"读取已有音频文件"即可。下一章 ASR 小节会演示这种方式。

---

## 本节练习

**练习 1（选择）**：`audio.get_wav_data(convert_rate=16000, convert_width=2)` 中，`convert_width=2` 的含义是？
- A. 立体声（2 声道）
- B. 16bit 位深（每个采样点 2 字节）
- C. 2 倍音量
- D. 录音 2 秒

**练习 2（填空）**：`adjust_for_ambient_noise()` 的作用是 ______，正确调用时机是 ______。

**练习 3（简答）**：为什么CANNLab 云环境无法做麦克风录音？有什么替代方案？

> 💡 参考答案见下方 code cell。

In [ ]:
# 查看本节练习答案
!cat ./answer/02.02_speech_basics/answers_recording.txt
